# SCIP

### Connecting to the shared baseline environment in Jupyter
Run these commands from the repository root.

```sh
conda env create -f src/baseline/environment.yml
conda activate rl4ising-baseline
python -m ipykernel install --user --name rl4ising-baseline --display-name "RL4Ising Baselines"
```

In [1]:
from pyscipopt import Model, quicksum
import networkx as nx

In [2]:
def read_nxgraph(filename):
    graph = nx.Graph()
    with open(filename) as file:
        lines = (line for line in file if line.strip() and not line.lstrip().startswith("//"))
        num_nodes, num_edges = map(int, next(lines).split())
        graph.add_nodes_from(range(num_nodes))
        for line in lines:
            node1, node2, weight = line.split()
            graph.add_edge(int(node1) - 1, int(node2) - 1, weight=float(weight))
    return graph

In [3]:
def solve(graph):
    model = Model("ising_qubo")
    x = {node: model.addVar(vtype="B", name=f"x_{i}")
            for i, node in enumerate(graph.nodes)}

    # y = x[node1] * x[node2]
    energy_terms = []
    for i, (node1, node2, data) in enumerate(graph.edges(data=True)):
        weight = float(data["weight"])
        y = model.addVar(vtype="B", name=f"y_{i}")
        model.addConsAnd([x[node1], x[node2]], y)
        energy_terms.append(
            -weight * (4 * y - 2 * x[node1] - 2 * x[node2] + 1)
        )

    model.setObjective(quicksum(energy_terms), "minimize")
    model.setParam("limits/time", 3600)
    model.setParam("limits/gap", 0.0)
    model.optimize()
    solution = model.getBestSol()
    if solution is None:
        status = model.getStatus()
        model.freeProb()
        raise RuntimeError(f"No solution found: {status}")

    objective = model.getObjVal()
    bits = "".join(str(round(model.getSolVal(solution, v))) for v in x.values())
    model.freeProb()
    return objective, bits

In [4]:
graph = read_nxgraph("../tutorials/VCA/dataset/EA/EA_10x10/10x10_uniform_seed1.txt")
objective, solution = solve(graph)
print("Objective:", objective)
print("Solution:", solution)

Objective: -78.65040168400627
Solution: 1000110110100111101100001000101011101100010110011001011010100110111011010001010101111111010100011110
